# Photo to Emoji — Train on GPU (Colab / Kaggle)

Run every cell top to bottom. On **Colab**: `Runtime > Change runtime type > T4 GPU`.
On **Kaggle**: `Settings > Accelerator > GPU T4 x2`.

At the end you'll get a `model.h5` file to download — that's the only file
you need to bring back to your local `gui.py`.

## 1. Check GPU is active

In [ ]:
import tensorflow as tf
print("GPU available:", tf.config.list_physical_devices('GPU'))

## 2. Download the FER-2013 dataset

- On **Kaggle**: `kagglehub` works with zero setup.
- On **Colab**: you need a Kaggle API token (`kaggle.json`). Run the cell below
  and upload it when prompted — get it from https://www.kaggle.com/settings
  ("Create New Token").

In [ ]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Upload your kaggle.json (from kaggle.com/settings) when prompted
    from google.colab import files
    print("Upload your kaggle.json file:")
    uploaded = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    for fname in uploaded:
        os.rename(fname, '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)

!pip install -q kagglehub
import kagglehub

dataset_path = kagglehub.dataset_download("msambare/fer2013")
print("Dataset downloaded to:", dataset_path)
print(os.listdir(dataset_path))

In [ ]:
# Locate the train/ and test/ subfolders inside whatever was downloaded
import glob

train_dir = glob.glob(os.path.join(dataset_path, '**', 'train'), recursive=True)[0]
val_dir = glob.glob(os.path.join(dataset_path, '**', 'test'), recursive=True)[0]

print("train_dir:", train_dir)
print("val_dir:  ", val_dir)
print("classes:", sorted(os.listdir(train_dir)))

## 3. Data generators

In [ ]:
from keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(rescale=1.0 / 255)
val_datagen = ImageDataGenerator(rescale=1.0 / 255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(48, 48),
    batch_size=64,
    color_mode="grayscale",
    class_mode="categorical",
)

validation_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(48, 48),
    batch_size=64,
    color_mode="grayscale",
    class_mode="categorical",
)

print("Class indices (folder -> label index) — keep this for gui.py:")
print(train_generator.class_indices)

num_train_imgs = train_generator.samples
num_val_imgs = validation_generator.samples

## 4. Build the CNN

In [ ]:
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D
from keras.optimizers import Adam

emotion_model = Sequential()

emotion_model.add(Conv2D(32, kernel_size=(3, 3), activation="relu", input_shape=(48, 48, 1)))
emotion_model.add(Conv2D(64, kernel_size=(3, 3), activation="relu"))
emotion_model.add(MaxPooling2D(pool_size=(2, 2)))
emotion_model.add(Dropout(0.25))

emotion_model.add(Conv2D(128, kernel_size=(3, 3), activation="relu"))
emotion_model.add(MaxPooling2D(pool_size=(2, 2)))
emotion_model.add(Conv2D(128, kernel_size=(3, 3), activation="relu"))
emotion_model.add(MaxPooling2D(pool_size=(2, 2)))
emotion_model.add(Dropout(0.25))

emotion_model.add(Flatten())
emotion_model.add(Dense(1024, activation="relu"))
emotion_model.add(Dropout(0.5))
emotion_model.add(Dense(7, activation="softmax"))

emotion_model.summary()

try:
    optimizer = Adam(learning_rate=0.0001, weight_decay=1e-6)
except TypeError:
    optimizer = Adam(lr=0.0001, decay=1e-6)

emotion_model.compile(
    loss="categorical_crossentropy",
    optimizer=optimizer,
    metrics=["accuracy"],
)

## 5. Train

On a T4 GPU this typically takes ~15–30 min for 50 epochs (vs. several hours
on CPU). Lower `EPOCHS` below for a quicker test run.

In [ ]:
EPOCHS = 50
BATCH_SIZE = 64

emotion_model_info = emotion_model.fit(
    train_generator,
    steps_per_epoch=num_train_imgs // BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=validation_generator,
    validation_steps=num_val_imgs // BATCH_SIZE,
)

## 6. Save weights and download

In [ ]:
emotion_model.save_weights("model.h5")
print("Saved model.h5")

if IN_COLAB:
    from google.colab import files
    files.download("model.h5")
else:
    # On Kaggle: model.h5 will appear in the notebook's Output panel
    # (right sidebar) once you commit/run the notebook — download it from there.
    print("On Kaggle: find model.h5 in the Output panel on the right and download it.")

## 7. (Optional) Plot training curves

In [ ]:
import matplotlib.pyplot as plt

history = emotion_model_info.history

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['accuracy'], label='train')
axes[0].plot(history['val_accuracy'], label='val')
axes[0].set_title('Accuracy')
axes[0].legend()

axes[1].plot(history['loss'], label='train')
axes[1].plot(history['val_loss'], label='val')
axes[1].set_title('Loss')
axes[1].legend()

plt.show()

---
### Next steps
1. Download `model.h5` from this notebook.
2. Place it in your local `photo-to-emoji/` project root (same folder as `gui.py`).
3. Run `python gui.py` locally to use your webcam with the trained model.
4. Double-check the `class_indices` printed in step 3 above match the
   `emotion_dict` order in `gui.py` (angry=0, disgust=1, fear=2, happy=3,
   neutral=4, sad=5, surprise=6).